In [1]:
# Ensure terrahydro and crest can be imported
%cd -q ..

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from pathlib import Path
from tqdm import tqdm
import tensorflow as tf
import numpy as np

from crest.archiver.Archiver import Archiver
from crest.data_server.DataServer import DataServer
from crest.data.loading import Datafile, Dataset
from crest.data import Batcher
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.Model import Model
from crest.utils.save_node_class import *
from lib.sm_model import *

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/tensorflow/python/keras/engine/training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


In [3]:
# For reproducibility
tf.keras.utils.set_random_seed(812)
tf.config.experimental.enable_op_determinism()

#### Soil moisture model


In [4]:
ROOT_PATH = Path(os.path.join("data","soil_moisture"))

locations = {
    'ERA5'       : ROOT_PATH.joinpath('ERA5.zarr'),
    'SMAP'       : ROOT_PATH.joinpath('SMAP.zarr'),
    'Soil'       : ROOT_PATH.joinpath('StaticAttributes.zarr/Soil'),
    'Irrigation' : ROOT_PATH.joinpath('StaticAttributes.zarr/Irrigation'),
    
}

# Define data attributes
features = {
    'ERA5'       : ['cape', 'cp', 'csfr','cvh', 'cvl', 'e', 'es',
                    'fal','lai_hv', 'lai_lv', 'msdwlwrf', 'msdwswrf',
                    'pev', 'sd','skt','smlt','sp', 'sro','slhf',
                    'sshf','ssro','stl1', 'stl2', 'stl3', 'stl4',
                    't2m','tp','u10', 'v10', 'z'],
    'Irrigation' : ['Actual', 'Equipped', 'GroundWater', 
                    'Non-Conventional', 'SurfaceWater'],
    'Soil'       : ['Clay_frac', 'Sand_frac', 'Silt_frac'],
    'SMAP'       : ['soil_moisture']
}

# Input and output features
INP = features['ERA5'] + features['Soil'] + features['Irrigation']
OUT = features['SMAP']

# Define depth and temporal extents
depth = {
    'ERA5': {'datetime': (5,0), 'latitude': 0, 'longitude': 0},
    'Irrigation': {'latitude': 0, 'longitude': 0},
    'Soil': {'latitude': 0, 'longitude': 0},
    'SMAP': {'datetime': 0, 'latitude': 0, 'longitude':0},
    }

extent_train = {'SMAP':{'datetime': [np.datetime64('2015-05-14 00:00:00'), np.datetime64('2015-05-29 23:00:01')]},
                'ERA5': {},
                'Soil': {},
                'Irrigation': {}
                }

extent_test = {'SMAP':{'datetime': [np.datetime64('2015-05-30 00:00:00'), np.datetime64('2015-05-30 23:00:01')]},
                'ERA5': {},
                'Soil': {},
                'Irrigation': {}
                }

dataset_train = Dataset([
                Datafile(
                    location     = locations[source],
                    features     = features[source],
                    window_depth = depth[source],
                    extent=extent_train[source],
                ) for source in features])

batch_train = Batcher(dataset_train, **{
    'batch_size' : 10,
    'features'   : [INP , OUT],
    'repeat'     : True,
    'numblocks'  : [1,1,1],
    'task_bytes' : 1e4,
    'workers'    : 0,
    'seed'       : 46
})

dataset_test = Dataset([
                Datafile(
                    location     = locations[source],
                    features     = features[source],
                    window_depth = depth[source],
                    extent=extent_test[source],
                ) for source in features])

# Use validation set to test
batch_valid = Batcher(dataset_test, **{
    'batch_size' : 10,
    'features'   : [INP , OUT],
    'repeat'     : True,
    'numblocks'  : [1,1,1],
    'task_bytes' : 1e4,
    'workers'    : 0,
    'seed'       : 46
})

batch_test = Batcher(dataset_test, **{
    'batch_size' : 10,
    'features'   : [INP , OUT],
    'repeat'     : False,
    'numblocks'  : [1,1,1],
    'task_bytes' : 1e4,
    'workers'    : 0,
    'shuffle'    : False
})

#### Create HTG basenode


In [5]:
inputs = {f: tf.TensorSpec((None,5, 1, 1)) 
        if f in features['ERA5'] else tf.TensorSpec((None,1, 1)) for f in INP}
outputs={f: tf.TensorSpec((None,1)) for f in OUT}

sm = sm_model()

htg = HierarchalTensorGraph(
    sm,
    name='SM',
    inputs=inputs,
    outputs=outputs
)

# Create model
model = Model(htg)

INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph SM
INFO:crest.model.Model:Initializing CREST Model
INFO:crest.model.HierarchalTensorGraph:Calling HTG SM
INFO:crest.model.HierarchalTensorGraph:Checking if HTG SM is a basenode
INFO:crest.model.HierarchalTensorGraph:Representing HTG SM
INFO:crest.model.HierarchalTensorGraph:Applying feature map to input in HTG SM
INFO:crest.model.HierarchalTensorGraph:Checking if HTG SM is a basenode
INFO:crest.model.HierarchalTensorGraph:Representing HTG SM
INFO:crest.model.HierarchalTensorGraph:Applying feature map to output in HTG SM
INFO:crest.model.Model:Completed Initializing CREST Model


In [6]:
model.compile(**{
    'optimizer' : 'Adam', 
    'loss'      : 'mse', 
    'metrics'   : [tf.keras.metrics.RootMeanSquaredError()]
})

model.fit(
    batch_train, 
    validation_data=batch_valid, 
    epochs=3, 
    steps_per_epoch=7, 
    validation_steps=1)

INFO:crest.model.Model:Compiling Keras Model
INFO:crest.model.Model:Building Keras Model
INFO:crest.model.Model:Training Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


Epoch 1/3


2025-05-14 08:48:16.292817: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


7/7 [==============================] - 5s 778ms/step - loss: 0.7183 - root_mean_squared_error: 0.8475 - val_loss: 0.0462 - val_root_mean_squared_error: 0.2150
Epoch 2/3
7/7 [==============================] - 3s 446ms/step - loss: 0.4573 - root_mean_squared_error: 0.6763 - val_loss: 0.0429 - val_root_mean_squared_error: 0.2072
Epoch 3/3
7/7 [==============================] - 3s 534ms/step - loss: 0.9499 - root_mean_squared_error: 0.9746 - val_loss: 0.0350 - val_root_mean_squared_error: 0.1870


In [7]:
res = model.evaluate(batch_test,return_dict=True)

INFO:crest.model.Model:Evaluating Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


1/1 [==============================] - 0s 34ms/step - loss: 0.0239 - root_mean_squared_error: 0.1545


In [8]:
model.save(os.path.join('outputs', 'soil_moisture_cache'))

INFO:crest.model.Model:Save Keras Model
INFO:crest.model.HierarchalTensorGraph:Serializing HTG SM to JSON
INFO:crest.model.HierarchalTensorGraph:Checking if HTG SM is a basenode
INFO:crest.model.HierarchalTensorGraph:Serializing basenode SM to JSON
INFO:crest.utils.save_node_class:write_pkl uses the root of the file path to also save obj metadata.
INFO:crest.utils.save_node_class:Saved metadata to outputs/metadata.json
INFO:crest.utils.save_node_class:Saved pickled object to outputs/sm_model.pkl
INFO:crest.model.Model:Save model as custom_model
INFO:crest.model.Model:Saving model with tensorflow keras.


INFO:tensorflow:Assets written to: outputs/soil_moisture_cache/assets


INFO:tensorflow:Assets written to: outputs/soil_moisture_cache/assets


In [9]:
soil_model = Model.load(os.path.join('outputs', 'soil_moisture_cache'), 'keras')

INFO:crest.utils.save_node_class:Found metadata at: outputs/metadata.json
INFO:crest.utils.save_node_class:Successfully unpickled object.
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph SM
INFO:crest.model.Model:Initializing CREST Model
INFO:crest.model.HierarchalTensorGraph:Calling HTG SM
INFO:crest.model.HierarchalTensorGraph:Checking if HTG SM is a basenode
INFO:crest.model.HierarchalTensorGraph:Representing HTG SM
INFO:crest.model.HierarchalTensorGraph:Applying feature map to input in HTG SM
INFO:crest.model.HierarchalTensorGraph:Checking if HTG SM is a basenode
INFO:crest.model.HierarchalTensorGraph:Representing HTG SM
INFO:crest.model.HierarchalTensorGraph:Applying feature map to output in HTG SM
INFO:crest.model.Model:Completed Initializing CREST Model
INFO:crest.model.Model:Load custom model from outputs
INFO:crest.model.Model:Loading model using Tensorflow Keras library.


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

ROOT_PATH = Path(os.path.join("data", "toy_dataset_zarr"))

features = {
    'ERA5'       : ['cape', 'cp', 'csfr','cvh', 'cvl', 'e', 'es',
                    'fal','lai_hv', 'lai_lv', 'msdwlwrf', 'msdwswrf',
                    'pev', 'sd','skt','smlt','sp', 'sro','slhf',
                    'sshf','ssro','stl1', 'stl2', 'stl3', 'stl4',
                    't2m','tp','u10', 'v10', 'z'],
    'Irrigation' : ['Actual', 'Equipped', 'GroundWater', 
                    'Non-Conventional', 'SurfaceWater'],
    'Soil'       : ['Clay_frac', 'Sand_frac', 'Silt_frac'],
}

locations = {
    'ERA5'       : ROOT_PATH.joinpath('ERA5.zarr'),
    'Soil'       : ROOT_PATH.joinpath('StaticAttributes.zarr/Soil'),
    'Irrigation' : ROOT_PATH.joinpath('StaticAttributes.zarr/Irrigation'),
}

INP = features['ERA5'] + features['Soil'] + features['Irrigation']

# Define depth and temporal extents
depth = {
    'ERA5': {'datetime': (4,0), 'latitude': 0, 'longitude': 0},
    'Irrigation': {'latitude': 0, 'longitude': 0},
    'Soil': {'latitude': 0, 'longitude': 0},
    'SMAP': {'datetime': 0, 'latitude': 0, 'longitude':0},
    }

extent_predict = {
                'ERA5': {'datetime': ('2018-05-21T00:00:00.000000000','2018-05-21T06:00:00.000000000')},
                'Soil': {},
                'Irrigation': {}}

# df = Datafile("toy_dataset_zarr/StaticAttributes.zarr", region=['MD', 'WA'])
# dataset_predict = Dataset([df])

dataset_predict = Dataset([
                Datafile(
                    location     = locations[source],
                    features     = features[source],
                    window_depth = depth[source],
                    extent=extent_predict[source],
                    region=['Maryland', 'Washington State', 'New Hampshire']
                ) for source in features])

#df.data.sel(features="Clay_frac").plot()

dataset_predict

In [ ]:
cache_kwargs = {
    'train': {'numblocks': (1,2,2),
    'cache_dir': 'caches/',
    'overwrite': False},
}

dataset_predict.cache(**cache_kwargs['train'])

In [ ]:
import tlz
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

batch_predict = Batcher(dataset_predict, **{
    'batch_size' : 1024,
    # 'features'   : [INP],
    'repeat'     : True,
    'shuffle'    : False,
    'numblocks'  : [1,1,1],
    'workers'    : 4,
    'seed'       : 46
})

# specify the output file to insert the predictions into
output_dir = os.path.join('outputs', 'model_predictions.zarr')

# the datafile to use as a the reference 
# structure for inserting the predictions into
data_schema = dataset_predict[0]

coordinates = ['datetime', 'latitude', 'longitude']

with Archiver(output_path = output_dir,
              coords = coordinates,
              data_schema = data_schema,
              overwrite=True) as a:
    with batch_predict as batcher:

        for b in batcher:
            # get the input variables from each sample
            data = tlz.merge_with(np.array, [s.to_dict(soil_model.graph.inputs) for s in b])

            # predict the output for each sample and remove extra axes
            pred = soil_model.predict_on_batch(data)
            # pred = {k: v for k, v in pred.items()}
            
            # capture the coordinates info
            coords = tlz.merge_with(np.array, [{c : s.coords[c][0][-1:] for c in coordinates} for s in b])
            
            # merge predictions and coordinates into one dict
            pred = pred | coords
                        
            # archive predictions
            a.archive(pred)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

output_data = xr.load_dataset(output_dir)

output_data.isel({'datetime': -2})['soil_moisture'].plot()